# ECG 4-class model (N / A / O / ~) — PhysioNet CinC 2017
Run cells top to bottom. Use a **GPU runtime** (Runtime > Change runtime type > T4). Outputs: `ecg_model.pt` + `ecg_core.py` (both go into your FastAPI backend).

In [ ]:
!wget -q -nc https://physionet.org/files/challenge-2017/1.0.0/training2017.zip
!unzip -q -n training2017.zip
import os; print(os.listdir('training2017')[:5])

In [ ]:
%%writefile ecg_core.py
"""Shared ECG code: preprocessing + model + inference. Import this in Colab AND in FastAPI."""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from math import gcd
from scipy.signal import butter, sosfiltfilt, resample_poly

FS = 300            # model sampling rate (Hz)
WIN = 9000          # 30 s window
CLASSES = ["N", "A", "O", "~"]
_SOS = butter(4, [0.5, 40], btype="band", fs=FS, output="sos")


def preprocess(x, fs=FS):
    """Raw 1-D ECG at `fs` Hz -> filtered, z-scored float32 at 300 Hz."""
    x = np.asarray(x, dtype=np.float64)
    if int(fs) != FS:
        g = gcd(int(fs), FS)
        x = resample_poly(x, FS // g, int(fs) // g)
    x = x - x.mean()
    x = sosfiltfilt(_SOS, x)
    x = (x - x.mean()) / (x.std() + 1e-8)
    return x.astype(np.float32)


def tile_pad(x, n=WIN):
    if len(x) >= n:
        return x
    return np.tile(x, int(np.ceil(n / len(x))))[:n]


def windows(x, n=WIN, stride=WIN // 2):
    x = tile_pad(x, n)
    starts = list(range(0, len(x) - n + 1, stride))
    if starts[-1] != len(x) - n:
        starts.append(len(x) - n)
    return [x[s:s + n] for s in starts]


class SE(nn.Module):
    def __init__(self, c, r=8):
        super().__init__()
        self.fc1 = nn.Linear(c, c // r)
        self.fc2 = nn.Linear(c // r, c)

    def forward(self, x):
        s = x.mean(-1)
        s = torch.sigmoid(self.fc2(F.relu(self.fc1(s))))
        return x * s.unsqueeze(-1)


class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv1d(cin, cout, 7, stride, 3, bias=False)
        self.b1 = nn.BatchNorm1d(cout)
        self.c2 = nn.Conv1d(cout, cout, 7, 1, 3, bias=False)
        self.b2 = nn.BatchNorm1d(cout)
        self.se = SE(cout)
        self.sc = None
        if stride != 1 or cin != cout:
            self.sc = nn.Sequential(nn.Conv1d(cin, cout, 1, stride, bias=False), nn.BatchNorm1d(cout))

    def forward(self, x):
        y = F.relu(self.b1(self.c1(x)))
        y = self.se(self.b2(self.c2(y)))
        return F.relu(y + (x if self.sc is None else self.sc(x)))


class ECGNet(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(1, 32, 15, 2, 7, bias=False), nn.BatchNorm1d(32), nn.ReLU())
        cfg = [(32, 32, 2), (32, 64, 2), (64, 64, 2), (64, 128, 2), (128, 128, 2), (128, 128, 2)]
        self.blocks = nn.Sequential(*[Block(*c) for c in cfg])
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(256, n_classes)

    def forward(self, x):                      # x: (B, 1, WIN)
        x = self.blocks(self.stem(x))
        x = torch.cat([x.mean(-1), x.amax(-1)], 1)
        return self.fc(self.drop(x))


@torch.no_grad()
def predict(model, raw_signal, fs=FS, device="cpu"):
    """Raw ECG array (any length >= ~2 s) at `fs` Hz -> label + probabilities."""
    model.eval()
    x = preprocess(raw_signal, fs)
    w = torch.from_numpy(np.stack(windows(x)))[:, None].to(device)
    p = F.softmax(model(w), 1).mean(0).cpu().numpy()
    return {"label": CLASSES[int(p.argmax())], "probs": {c: float(v) for c, v in zip(CLASSES, p)}}


def load_model(path, device="cpu"):
    ck = torch.load(path, map_location=device)
    m = ECGNet(len(ck["classes"]))
    m.load_state_dict(ck["state_dict"])
    return m.to(device).eval()

In [ ]:
import glob, random, hashlib, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from scipy.io import loadmat
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from ecg_core import *

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# ---- load labels + signals ----
ref = pd.read_csv("training2017/REFERENCE.csv", header=None, names=["name", "label"])
assert len(ref) == 8528, len(ref)
cmap = {c: i for i, c in enumerate(CLASSES)}
y_all = ref["label"].map(cmap).values
sigs_all = [preprocess(loadmat(f"training2017/{n}.mat")["val"][0].astype(np.float64)) for n in ref["name"]]
print("loaded", len(sigs_all), "class counts:", np.bincount(y_all))

In [ ]:
# ---- split BY RECORDING: 70 / 15 / 15, stratified (test set is never used for tuning) ----
idx = np.arange(len(y_all))
tr, tmp = train_test_split(idx, test_size=0.30, stratify=y_all, random_state=SEED)
va, te = train_test_split(tmp, test_size=0.50, stratify=y_all[tmp], random_state=SEED)
sub = lambda ix: ([sigs_all[i] for i in ix], y_all[ix])
(Xtr, ytr), (Xva, yva), (Xte, yte) = sub(tr), sub(va), sub(te)
print(len(tr), len(va), len(te))

class DS(Dataset):
    def __init__(self, X, y, train): self.X, self.y, self.train = X, y, train
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = tile_pad(self.X[i])
        s = np.random.randint(0, len(x) - WIN + 1)
        x = x[s:s + WIN].copy()
        if self.train:
            x *= np.random.uniform(0.7, 1.3)
            x += np.random.randn(WIN) * np.random.uniform(0, 0.15)                       # sensor noise
            t = np.arange(WIN) / FS
            x += np.random.uniform(0, 0.6) * np.sin(2*np.pi*np.random.uniform(0.05, 0.5)*t + np.random.uniform(0, 6.28))  # baseline wander
        return torch.from_numpy(x.astype(np.float32))[None], int(self.y[i])

train_dl = DataLoader(DS(Xtr, ytr, True), batch_size=64, shuffle=True, num_workers=2, drop_last=True)

@torch.no_grad()
def evaluate(model, X, y):
    model.eval(); P = []
    for x in X:
        w = torch.from_numpy(np.stack(windows(x)))[:, None].to(device)
        P.append(F.softmax(model(w), 1).mean(0).cpu().numpy())
    pred = np.array(P).argmax(1)
    return f1_score(y, pred, average="macro"), pred

In [ ]:
# ---- train ----
EPOCHS = 40
model = ECGNet(len(CLASSES)).to(device)
cnt = np.bincount(ytr, minlength=4)
cw = torch.tensor((cnt.sum() / (4 * cnt)) ** 0.5, dtype=torch.float32, device=device)   # softened inverse-frequency
crit = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.05)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-3, epochs=EPOCHS, steps_per_epoch=len(train_dl))
use_amp = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

best, best_state = -1, None
for ep in range(EPOCHS):
    model.train(); tot = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        with torch.autocast(device_type="cuda", enabled=use_amp):
            loss = crit(model(xb), yb)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item()
    f1, _ = evaluate(model, Xva, yva)
    if f1 > best:
        best, best_state = f1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"epoch {ep+1:02d}  loss {tot/len(train_dl):.4f}  val macro-F1 {f1:.4f}  best {best:.4f}")

In [ ]:
# ---- final test on held-out recordings ----
model.load_state_dict(best_state)
f1, pred = evaluate(model, Xte, yte)
print("TEST macro-F1:", round(f1, 4))
print(classification_report(yte, pred, target_names=CLASSES, digits=3))
print("rows=true, cols=pred", CLASSES); print(confusion_matrix(yte, pred))

In [ ]:
# ---- save + verify the checkpoint reloads identically ----
torch.save({"state_dict": best_state, "classes": CLASSES, "fs": FS, "win": WIN}, "ecg_model.pt")
print("sha256:", hashlib.sha256(open("ecg_model.pt", "rb").read()).hexdigest())

m2 = load_model("ecg_model.pt", device)
probe = torch.from_numpy(np.stack(windows(Xte[0])))[:, None].to(device)
model.eval()
with torch.no_grad():
    assert torch.allclose(model(probe), m2(probe), atol=1e-5)
print("checkpoint reload OK")

# end-to-end inference on a RAW signal (this is exactly what FastAPI will call)
raw = loadmat(f"training2017/{ref['name'][te[0]]}.mat")["val"][0]
print("true:", CLASSES[yte[0]], "->", predict(m2, raw, fs=300, device=device))

In [ ]:
from google.colab import files
files.download("ecg_model.pt"); files.download("ecg_core.py")